# LangChain

In [6]:
!pip cache purge --quiet

In [7]:
!pip install langchain==0.3.27 \
             langchain-community==0.3.25 \
             langchain-openai==0.3.33 \
             langchain-singlestore==1.0.0 --quiet

In [8]:
import logging
import pandas as pd
import time
import warnings

from langchain.agents import initialize_agent, Tool
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.memory import ConversationBufferMemory
from langchain.prompts import ChatPromptTemplate
from langchain_community.agent_toolkits import create_sql_agent
from langchain_community.utilities import SQLDatabase
from langchain_core.documents import Document
from langchain_core.globals import get_llm_cache, set_llm_cache
from langchain_core.outputs import Generation
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_singlestore import SingleStoreChatMessageHistory, SingleStoreSemanticCache
from langchain_singlestore.vectorstores import DistanceStrategy, SingleStoreVectorStore
from singlestoredb.management import get_secret
from sqlalchemy.exc import SAWarning, SQLAlchemyError

In [9]:
warnings.filterwarnings("ignore", category = DeprecationWarning)
warnings.filterwarnings("ignore", category = SAWarning)

# Only log errors for key libraries
logging.getLogger("httpx").setLevel(logging.ERROR)
logging.getLogger("langchain").setLevel(logging.ERROR)
logging.getLogger("sqlalchemy").setLevel(logging.ERROR)

In [10]:
LLM_MODEL = "gpt-4.1-mini"
EMBEDDING_MODEL = "text-embedding-3-small"

os.environ["OPENAI_API_KEY"] = get_secret("OPENAI_API_KEY")

In [11]:
llm = ChatOpenAI(
    model = LLM_MODEL,
    temperature = 0
)

## Natural Language

<div class="alert alert-block alert-warning">
    <b class="fa fa-solid fa-exclamation-circle"></b>
    <div>
        <p><b>Action Required</b></p>
        <p>Select the database from the drop-down menu at the top of this notebook. It updates the <b>connection_url</b> which is used by SQLAlchemy to make connections to the selected database.</p>
    </div>
</div>

In [12]:
try:
    db = SQLDatabase.from_uri(
        connection_url,
        include_tables = ["tick"]
    )
except SQLAlchemyError as e:
    print(f"Error connecting to the database: {e}")
    exit()

In [13]:
sql_agent_executor = create_sql_agent(
    llm = llm,
    db = db,
    agent_type = "openai-tools",
    verbose = False
)

In [14]:
def query_tick(question: str) -> str:
    """
    Ask a natural language question to the SQL agent and return only the text output.
    """
    try:
        # Use the dedicated SQL agent executor
        result = sql_agent_executor.invoke({"input": question})
        return result.get("output", "").strip()
    except Exception as e:
        return f"Tool Error: Could not execute SQL query: {e}"

tick_tool = Tool(
    name = "TickTable",
    func = query_tick,
    description = "Use this to query the tick table in natural language about stock prices, volumes and time."
)

In [15]:
print(query_tick(
    "Show me the last 5 ticks for BBRQ-FX. Present the result as a table."
    )
)

Here are the last 5 ticks for BBRQ-FX:

| ts                  | open    | high    | low     | close   | volume   |
|---------------------|---------|---------|---------|---------|----------|
| 2015-11-24 00:00:00 | 999.83  | 1017.41 | 985.60  | 1004.39 | 2066982  |
| 2015-11-23 00:00:00 | 1020.74 | 1039.59 | 1010.85 | 1019.04 | 2005610  |
| 2015-11-20 00:00:00 | 991.68  | 992.87  | 977.93  | 990.32  | 1995583  |
| 2015-11-19 00:00:00 | 1001.18 | 1008.44 | 992.92  | 1003.77 | 33279776 |
| 2015-11-18 00:00:00 | 1037.09 | 1052.68 | 1023.30 | 1042.63 | 5600879  |


In [16]:
print(query_tick(
    "Which ticker had the highest close price?"
    )
)

The ticker with the highest close price is WVVF-FX with a close price of 6301.59.


In [17]:
print(query_tick(
    "What is the average trading volume for BJBY-FX?"
    )
)

The average trading volume for BJBY-FX is approximately 6,702,839.


In [18]:
print(query_tick(
    "Using the latest timestamp in the tick table as the reference, "
    "return all tickers from the 10 seconds before that timestamp "
    "where the close price is above 500 and sort the results alphabetically by ticker."
    )
)

The tickers from the 10 seconds before the latest timestamp in the tick table where the close price is above 500, sorted alphabetically, are:

1. BBRQ-FX
2. BBYX-FX
3. BKCZ-FX
4. BMJH-FX
5. BPDZ-FX
6. BPKV-FX
7. BTSP-FX
8. CCMN-FX
9. CGHY-FX
10. CHWP-FX

All these ticks have the latest timestamp of 2015-11-24 00:00:00.


## RAG

In [19]:
company_info = {
    "BBRQ-FX": "BBRQ Corp is a global technology conglomerate known for its enterprise software platforms, cloud infrastructure services and a dominant position in business productivity tools.",
    "BJBY-FX": "BJBY Dynamics is a consumer electronics and devices company renowned for its flagship smartphone line, wearable technology and a fast-growing digital services ecosystem.",
    "YWMG-FX": "YWMG Group is an e-commerce and logistics powerhouse with expanding interests in cloud computing, digital advertising and subscription-based media streaming.",
    "HRPD-FX": "HRPD Technologies is a semiconductor and hardware manufacturer supplying chips and processing units to the automotive, aerospace and artificial intelligence industries."
}

In [20]:
documents = [
    Document(page_content = text, metadata = {"symbol": sym})
    for sym, text in company_info.items()
]

In [21]:
# Use OpenAI embeddings
embeddings = OpenAIEmbeddings(
    model = EMBEDDING_MODEL
)

# Vector dimension is fixed per model, but can still check dynamically
dimensions = len(embeddings.embed_query("test"))

<div class="alert alert-block alert-warning">
    <b class="fa fa-solid fa-exclamation-circle"></b>
    <div>
        <p><b>Action Required</b></p>
        <p>Select the database from the drop-down menu at the top of this notebook. It updates the <b>connection_url</b> which is used by SQLAlchemy to make connections to the selected database.</p>
    </div>
</div>

In [22]:
from sqlalchemy import *

db_connection = create_engine(connection_url)

In [23]:
with db_connection.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS company_knowledge;"))

In [24]:
vector_store = SingleStoreVectorStore(
    embeddings,
    table_name = "company_knowledge",
    distance_strategy = "DOT_PRODUCT",
    use_vector_index = True,
    vector_size = dimensions
)

vector_store.add_documents(documents);

In [25]:
df = pd.read_sql(
    """
    SELECT LEFT(content, 30) AS content, LEFT(vector :> JSON, 30) AS vector, metadata::symbol AS metadata
    FROM company_knowledge
    """,
    db_connection
)

In [26]:
df.head()

,content,vector,metadata
0,HRPD Technologies is a semicon,"[-0.010345459,-0.0561523438,0.",HRPD-FX
1,BJBY Dynamics is a consumer el,"[0.0344238281,0.000962734222,0",BJBY-FX
2,BBRQ Corp is a global technolo,"[0.0343017578,-0.0219573975,0.",BBRQ-FX
3,YWMG Group is an e-commerce an,"[-0.00539016724,0.0135269165,0",YWMG-FX


In [27]:
print(vector_store.similarity_search(
    "What are the most popular consumer devices and services that BBRQ sells?",
    k = 1
)[0].page_content)

BBRQ Corp is a global technology conglomerate known for its enterprise software platforms, cloud infrastructure services and a dominant position in business productivity tools.


In [28]:
print(vector_store.similarity_search(
    "Describe BJBY's main contributions to consumer electronics.",
    k = 1
)[0].page_content)

BJBY Dynamics is a consumer electronics and devices company renowned for its flagship smartphone line, wearable technology and a fast-growing digital services ecosystem.


In [29]:
print(vector_store.similarity_search(
    "Can you detail the core technologies and services provided by YWMG?",
    k = 1
)[0].page_content)

YWMG Group is an e-commerce and logistics powerhouse with expanding interests in cloud computing, digital advertising and subscription-based media streaming.


In [30]:
print(vector_store.similarity_search(
    "What are the primary sectors in which HRPD operates?",
    k = 1
)[0].page_content)

HRPD Technologies is a semiconductor and hardware manufacturer supplying chips and processing units to the automotive, aerospace and artificial intelligence industries.


## Conversational Memory

In [31]:
print("\n--- In-Memory Agent (No Persistence) ---")

in_memory = ConversationBufferMemory(
    memory_key = "chat_history",
    return_messages = False
)

# Create a conversational agent with memory
agent_memory = initialize_agent(
    tools = [tick_tool],
    llm = llm,
    agent = "zero-shot-react-description",
    memory = in_memory,
    verbose = False,
    max_iterations = 6,
    handle_parsing_errors = True
)


--- In-Memory Agent (No Persistence) ---


In [32]:
print(agent_memory.invoke(
    "Show me the last 5 ticks for BBRQ-FX. Present the result as a table."
)["output"])

| Date       | Open    | High    | Low     | Close   | Volume     |
|------------|---------|---------|---------|---------|------------|
| 2015-11-24 | 999.83  | 1017.41 | 985.60  | 1004.39 | 2,066,982  |
| 2015-11-23 | 1020.74 | 1039.59 | 1010.85 | 1019.04 | 2,005,610  |
| 2015-11-20 | 991.68  | 992.87  | 977.93  | 990.32  | 1,995,583  |
| 2015-11-19 | 1001.18 | 1008.44 | 992.92  | 1003.77 | 33,279,776 |
| 2015-11-18 | 1037.09 | 1052.68 | 1023.30 | 1042.63 | 5,600,879  |


In [33]:
print(agent_memory.invoke(
    "Now compare BBRQ-FX with BJBY-FX for the last 5 ticks."
)["output"])

BBRQ-FX consistently shows higher prices than BJBY-FX over the last 5 ticks. Volume for BBRQ-FX is generally higher except on 2015-11-24 when BJBY-FX had more than double the volume.


In [35]:
print(agent_memory.invoke(
    "Which had the higher last close, BBRQ-FX or BJBY-FX?"
)["output"])

BBRQ-FX had the higher last close price.


In [36]:
print("\nCurrent conversation memory (in-memory buffer):\n")

for msg in in_memory.chat_memory.messages:
    # Use .type to get 'human' or 'ai'
    role = msg.type.upper()
    print(f"{role}: {msg.content}")


Current conversation memory (in-memory buffer):

HUMAN: Show me the last 5 ticks for BBRQ-FX. Present the result as a table.
AI: | Date       | Open    | High    | Low     | Close   | Volume     |
|------------|---------|---------|---------|---------|------------|
| 2015-11-24 | 999.83  | 1017.41 | 985.60  | 1004.39 | 2,066,982  |
| 2015-11-23 | 1020.74 | 1039.59 | 1010.85 | 1019.04 | 2,005,610  |
| 2015-11-20 | 991.68  | 992.87  | 977.93  | 990.32  | 1,995,583  |
| 2015-11-19 | 1001.18 | 1008.44 | 992.92  | 1003.77 | 33,279,776 |
| 2015-11-18 | 1037.09 | 1052.68 | 1023.30 | 1042.63 | 5,600,879  |
HUMAN: Now compare BBRQ-FX with BJBY-FX for the last 5 ticks.
AI: BBRQ-FX consistently shows higher prices than BJBY-FX over the last 5 ticks. Volume for BBRQ-FX is generally higher except on 2015-11-24 when BJBY-FX had more than double the volume.
HUMAN: Which had the higher last close, BBRQ-FX or BJBY-FX?
AI: BBRQ-FX had the higher last close price.


In [37]:
in_memory.clear()

<div class="alert alert-block alert-warning">
    <b class="fa fa-solid fa-exclamation-circle"></b>
    <div>
        <p><b>Action Required</b></p>
        <p>Select the database from the drop-down menu at the top of this notebook. It updates the <b>connection_url</b> which is used by SQLAlchemy to make connections to the selected database.</p>
    </div>
</div>

In [38]:
from sqlalchemy import *

db_connection = create_engine(connection_url)

In [39]:
def create_singlestore_agent_memory(
    session_id: str,
    table_name: str = "chat_history",
    clear_history: bool = True
) -> ConversationBufferMemory:
    """
    Consolidates the creation of SingleStore-backed memory for a LangChain Agent.

    Args:
        session_id: The unique ID for the conversation thread.
        table_name: The name of the table in SingleStore.
        clear_history: If True, clears the session's history before returning.

    Returns:
        A ConversationBufferMemory object ready for use in initialize_agent.
    """
    chat_history = SingleStoreChatMessageHistory(
        session_id = session_id,
        table_name = table_name,
    )

    if clear_history:
        chat_history.clear()

    memory = ConversationBufferMemory(
        memory_key = "chat_history",
        chat_memory = chat_history,
        return_messages = True
    )

    return memory

In [40]:
print("\n--- Persistent Agent (SingleStore Only) ---")

session_id = "persistent"
persistent_memory = create_singlestore_agent_memory(
    session_id = session_id,
    table_name = "chat_history",
    clear_history = True
)

# Create the agent using the persistent memory
agent_persistent = initialize_agent(
    tools = [tick_tool],
    llm = llm,
    agent = "zero-shot-react-description",
    memory = persistent_memory,
    verbose = False,
    max_iterations = 6,
    handle_parsing_errors = True
)


--- Persistent Agent (SingleStore Only) ---


In [41]:
print(agent_persistent.invoke(
    "Show me the last 5 ticks for BBRQ-FX. Present the result as a table."
)["output"])

| Date       | Open    | High    | Low     | Close   | Volume    |
|------------|---------|---------|---------|---------|-----------|
| 2015-11-24 | 999.83  | 1017.41 | 985.60  | 1004.39 | 2,066,982 |
| 2015-11-23 | 1020.74 | 1039.59 | 1010.85 | 1019.04 | 2,005,610 |
| 2015-11-20 | 991.68  | 992.87  | 977.93  | 990.32  | 1,995,583 |
| 2015-11-19 | 1001.18 | 1008.44 | 992.92  | 1003.77 | 33,279,776|
| 2015-11-18 | 1037.09 | 1052.68 | 1023.30 | 1042.63 | 5,600,879 |


In [42]:
print(agent_persistent.invoke(
    "Now compare BBRQ-FX with BJBY-FX for the last 5 ticks."
)["output"])

Over the last 5 ticks, BBRQ-FX has consistently higher prices (more than double) compared to BJBY-FX. Volume for BBRQ-FX is generally higher, notably on 2015-11-19 where it is significantly larger. BJBY-FX has lower prices and somewhat lower or comparable volumes on other days.


In [43]:
print(agent_persistent.invoke(
    "Which had the higher last close, BBRQ-FX or BJBY-FX?"
)["output"])

BBRQ-FX had the higher last close price.


In [45]:
with db_connection.connect() as conn:
    rows = conn.execute(
        text("SELECT id, session_id, message FROM chat_history WHERE session_id = :sid ORDER BY id"),
        {"sid": session_id}
    ).fetchall()

print(f"Current conversation memory (persistent-memory buffer) for session: '{session_id}':\n")
for r in rows:
    msg = r.message
    role = msg.get("type", "unknown").upper()
    content = msg.get("data", {}).get("content", "")
    print(f"{role}: {content}")

Current conversation memory (persistent-memory buffer) for session: 'persistent':

HUMAN: Show me the last 5 ticks for BBRQ-FX. Present the result as a table.
AI: | Date       | Open    | High    | Low     | Close   | Volume    |
|------------|---------|---------|---------|---------|-----------|
| 2015-11-24 | 999.83  | 1017.41 | 985.60  | 1004.39 | 2,066,982 |
| 2015-11-23 | 1020.74 | 1039.59 | 1010.85 | 1019.04 | 2,005,610 |
| 2015-11-20 | 991.68  | 992.87  | 977.93  | 990.32  | 1,995,583 |
| 2015-11-19 | 1001.18 | 1008.44 | 992.92  | 1003.77 | 33,279,776|
| 2015-11-18 | 1037.09 | 1052.68 | 1023.30 | 1042.63 | 5,600,879 |
HUMAN: Now compare BBRQ-FX with BJBY-FX for the last 5 ticks.
AI: Over the last 5 ticks, BBRQ-FX has consistently higher prices (more than double) compared to BJBY-FX. Volume for BBRQ-FX is generally higher, notably on 2015-11-19 where it is significantly larger. BJBY-FX has lower prices and somewhat lower or comparable volumes on other days.
HUMAN: Which had the hi

## NL and RAG

In [46]:
# Create a retriever from the vector store
company_retriever = vector_store.as_retriever(search_kwargs = {"k": 1})

# Define the RAG prompt (LCEL Standard)
rag_prompt = ChatPromptTemplate.from_template(
    """
    You are a helpful assistant. Use the following context to answer the user's question.
    If the context does not contain the answer, politely state that the information is not in your knowledge base.

    Context: {context}

    Question: {input}
    """
)

# Create the document combination chain
document_chain = create_stuff_documents_chain(llm, rag_prompt)

# Create the RAG retrieval chain (Retriever + LLM)
rag_chain = create_retrieval_chain(company_retriever, document_chain)

In [47]:
def query_company_info_lcel(question: str) -> str:
    """Use this to get background info about a company."""
    # The rag_chain handles the retrieval and generation internally
    result = rag_chain.invoke({"input": question})
    return result["answer"]

company_tool = Tool(
    name = "CompanyKnowledge",
    func = query_company_info_lcel,
    description = "Use this to get background info about a company using its stock symbol or company name."
)

In [48]:
print("\n--- Natural Language and RAG Agent (Combined) ---")

session_id = "combined"
combined_memory = create_singlestore_agent_memory(
    session_id = session_id,
    table_name = "chat_history",
    clear_history = True
)

# Create the agent with both tools
agent_combined = initialize_agent(
    tools = [tick_tool, company_tool],
    llm = llm,
    agent = "zero-shot-react-description",
    memory = combined_memory,
    verbose = False,
    max_iterations = 6,
    handle_parsing_errors = True
)


--- Natural Language and RAG Agent (Combined) ---


In [49]:
print(agent_combined.invoke(
    "Show me the last 5 ticks for BBRQ-FX. Present the result as a table."
)["output"])

| Date       | Open    | High    | Low     | Close   | Volume     |
|------------|---------|---------|---------|---------|------------|
| 2015-11-24 | 999.83  | 1017.41 | 985.60  | 1004.39 | 2,066,982  |
| 2015-11-23 | 1020.74 | 1039.59 | 1010.85 | 1019.04 | 2,005,610  |
| 2015-11-20 | 991.68  | 992.87  | 977.93  | 990.32  | 1,995,583  |
| 2015-11-19 | 1001.18 | 1008.44 | 992.92  | 1003.77 | 33,279,776 |
| 2015-11-18 | 1037.09 | 1052.68 | 1023.30 | 1042.63 | 5,600,879  |


In [50]:
print(agent_combined.invoke(
    "What products does BJBY make?"
)["output"])

BJBY makes consumer electronics and devices, including a flagship smartphone line, wearable technology, and provides a fast-growing digital services ecosystem.


In [51]:
print(agent_combined.invoke(
    "Now do the same for HRPD"
)["output"])

HRPD Technologies is a semiconductor and hardware manufacturer supplying chips and processing units to the automotive, aerospace, and artificial intelligence industries. However, there is no recent stock price, volume, or time data available for HRPD in the tick table. If you need information on another company or symbol, please let me know.


## Semantic Caching

In [52]:
semantic_cache = SingleStoreSemanticCache(
    embedding = embeddings,
    search_threshold = 0.75,
    distance_strategy = DistanceStrategy.DOT_PRODUCT
)

set_llm_cache(semantic_cache)
print("SingleStore Semantic Cache initialized and set globally.")

SingleStore Semantic Cache initialized and set globally.


In [53]:
# Patch
def semantic_lookup(self, prompt: str, llm_string: str):
    """
    Lookup using embeddings similarity with exact-match fallback.
    Vector search is attempted first, then exact-match if needed.
    """
    llm_cache = self._get_llm_cache(llm_string)

    # Attempt vector search first
    try:
        vector = self.embedding.embed_query(prompt)
        results = llm_cache.similarity_search_by_vector(
            vector = vector,
            k = 1,
            embedding = self.embedding
        )
        if results:
            doc, score = results[0]
            if ((llm_cache.distance_strategy == DistanceStrategy.DOT_PRODUCT and score >= self.search_threshold) or
                (llm_cache.distance_strategy == DistanceStrategy.EUCLIDEAN_DISTANCE and score <= self.search_threshold)):
                return loads(doc.metadata["return_val"])
    except Exception as e:
        pass

    # Fallback to exact match
    for doc in getattr(llm_cache, "docs", []):
        if doc.page_content == prompt:
            return loads(doc.metadata["return_val"])

    return None

SingleStoreSemanticCache.lookup = semantic_lookup

In [54]:
# Function for cached LLM call
def timed_llm_call(llm, prompt, semantic_cache):
    llm_string = llm._get_llm_string()

    # Check cache using fast, optimized lookup patch
    cached_result = semantic_cache.lookup(prompt, llm_string)
    if cached_result:
        # Cache hit, so return immediately
        return cached_result[0].text, 0.0

    # Cache miss so time the actual LLM call
    start_time = time.time()
    result = llm.invoke(prompt)
    elapsed = time.time() - start_time

    # Cache update, use explicit update logic to save the result
    semantic_cache.update(
        prompt = prompt,
        llm_string = llm_string,
        return_val = [Generation(text = result.content)]
    )
    return result.content, elapsed

In [55]:
semantic_cache.clear(llm_string = llm._get_llm_string())
print("Cache cleared for a fresh run.")

Cache cleared for a fresh run.


In [56]:
print("\n--- First Call (Cache Miss) ---")
prompt1 = "Explain the key factors influencing BBRQ’s stock price."
result1, time1 = timed_llm_call(llm, prompt1, semantic_cache)
print(f"Prompt: {prompt1}\nTime: {time1:.2f}s\nResult: {result1[:80]}...")


--- First Call (Cache Miss) ---
Prompt: Explain the key factors influencing BBRQ’s stock price.
Time: 7.11s
Result: To explain the key factors influencing BBRQ’s stock price, it’s important to con...


In [57]:
print("\n--- Second Call (Semantic Hit - Expect faster response) ---")
prompt2 = "What are the main reasons for BBRQ’s stock price moving up or down?"
result2, time2 = timed_llm_call(llm, prompt2, semantic_cache)
print(f"Prompt: {prompt2}\nTime: {time2:.2f}s\nResult: {result2[:80]}...")


--- Second Call (Semantic Hit - Expect faster response) ---
Prompt: What are the main reasons for BBRQ’s stock price moving up or down?
Time: 3.66s
Result: To provide an accurate answer about the main reasons for BBRQ’s stock price move...


In [58]:
print("\n--- Third Call (Cache Miss - Expect slow response) ---")
prompt3 = "What is the process of nuclear fusion?"
result3, time3 = timed_llm_call(llm, prompt3, semantic_cache)
print(f"Prompt: {prompt3}\nTime: {time3:.2f}s\nResult: {result3[:80]}...")


--- Third Call (Cache Miss - Expect slow response) ---
Prompt: What is the process of nuclear fusion?
Time: 6.19s
Result: Nuclear fusion is the process by which two light atomic nuclei combine to form a...
